Imports

In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## the cleaning


In [ ]:
athletes = pd.read_csv(r"../data/raw/athlete_events.csv")
regions = pd.read_csv(r"../data/raw/noc_regions.csv")

mask = regions["NOC"] == "SIN"
regions.loc[mask,"NOC"] = "SGP"

regions.loc[regions["region"].isna(), "region"] = regions["notes"]


## Member 3

### Investigating missing values

In [132]:

athletes = pd.read_csv(r"../data/raw/athlete_events.csv")
regions = pd.read_csv(r"../data/raw/noc_regions.csv")


df = athletes.merge(regions, on="NOC", how="left")

In [133]:
(df["region"].isna()).sum()

np.int64(370)

In [134]:
df.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN,China,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN,China,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN,Denmark,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold,Denmark,NaN
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN,Netherlands,NaN


In [168]:
missing = df.isna().sum()
missing = missing[missing > 0]

pct = round(df[missing.index].isna().mean() * 100,2)

summary = pd.DataFrame({
        "missing count": missing.map(lambda x: f"{x:,}"),
        "missing percentage": pct.astype(str) + "%"
    })
display(summary)

,missing count,missing percentage
Age,"9,474",3.49%
Height,"60,171",22.19%
Weight,"62,875",23.19%
Medal,"231,333",85.33%
region,370,0.14%
notes,"266,077",98.14%


### Investigating duplicates

In [139]:
dupes_count = df.duplicated().sum()
dupes_prct = round((df.duplicated().mean() * 100 ),2)

print(df.duplicated().sum())
print(round(df.duplicated().mean() * 100, 2), "%", sep="")

1385
0.51%


#### the filtered columns are columns with too many different values for there to be duplication or meaning behind the duplication

#### discoveries: 
All duplicates were in Summer olympics between Year 1900 and 1948
Sports that contained duplicates were art competitions, sailing, cycling, Equestrianism.
Art competitions: allowed multiple painting submission from a single athlete
sailing, cycling: multiple races in a single event. without race number, they seem identical.
Equestrianism: same participant rode multiple horses in the same event.


In [169]:
dups = df[df.duplicated(keep=False)]
for col in dups.columns:
    if col in ("ID", "Name", "Team", "NOC", "Medal", "Age", "Sex", "region", "Weight", "Height", "City", "Event", "notes", "Games"):
        continue
    print(f"--- {col} ---")
    print(dups[col].value_counts(dropna=False))
    print()

--- Year ---
Year
1932    692
1928    467
1936    377
1924    175
1948    146
1900     76
1908     64
Name: count, dtype: int64

--- Season ---
Season
Summer    1997
Name: count, dtype: int64

--- Sport ---
Sport
Art Competitions    1857
Sailing               74
Cycling               64
Equestrianism          2
Name: count, dtype: int64



### Investigating Unusual Values and inconsistencies


In [142]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 17 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
 15  region  270746 non-null  str    
 16  notes   5039 non-null    str    
dtypes: float64(3), int64(2), str(12)
memory usage: 35.2 MB


In [143]:

stats = df[["Age", "Height", "Weight", "Year"]].describe().round(1)
# display(stats)
stats[["Age", "Height", "Weight"]] = stats[["Age", "Height", "Weight"]].map(lambda x: f"{x:,}")
stats["Year"] = stats["Year"].astype(int).astype(str)
display(stats)

,Age,Height,Weight,Year
count,"261,642.0","210,945.0","208,241.0",271116
mean,25.6,175.3,70.7,1978
std,6.4,10.5,14.3,29
min,10.0,127.0,25.0,1896
25%,21.0,168.0,60.0,1960
50%,24.0,175.0,70.0,1988
75%,28.0,183.0,79.0,2002
max,97.0,226.0,214.0,2016


young age doesn't seem error

In [144]:
df.nsmallest(5,"Age")

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
142882,71691,Dimitrios Loundras,M,10.0,NaN,NaN,Ethnikos Gymnastikos Syllogos,GRE,1896 Summer,1896,Summer,Athina,Gymnastics,"Gymnastics Men's Parallel Bars, Teams",Bronze,Greece,NaN
43468,22411,Magdalena Cecilia Colledge,F,11.0,152.0,NaN,Great Britain,GBR,1932 Winter,1932,Winter,Lake Placid,Figure Skating,Figure Skating Women's Singles,NaN,UK,NaN
73461,37333,Carlos Bienvenido Front Barrera,M,11.0,NaN,NaN,Spain,ESP,1992 Summer,1992,Summer,Barcelona,Rowing,Rowing Men's Coxed Eights,NaN,Spain,NaN
79024,40129,Luigina Giavotti,F,11.0,NaN,NaN,Italy,ITA,1928 Summer,1928,Summer,Amsterdam,Gymnastics,Gymnastics Women's Team All-Around,Silver,Italy,NaN
94058,47618,"Sonja Henie (-Topping, -Gardiner, -Onstad)",F,11.0,155.0,45.0,Norway,NOR,1924 Winter,1924,Winter,Chamonix,Figure Skating,Figure Skating Women's Singles,NaN,Norway,NaN


old age doesn't seem error competes in arts

In [145]:
df.nlargest(5, "Age")

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,region,notes
257054,128719,John Quincy Adams Ward,M,97.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Sculpturing, Statues",NaN,USA,NaN
98118,49663,Winslow Homer,M,96.0,NaN,NaN,United States,USA,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,USA,NaN
60861,31173,Thomas Cowperthwait Eakins,M,88.0,NaN,NaN,United States,USA,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,USA,NaN
60862,31173,Thomas Cowperthwait Eakins,M,88.0,NaN,NaN,United States,USA,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,USA,NaN
60863,31173,Thomas Cowperthwait Eakins,M,88.0,NaN,NaN,United States,USA,1932 Summer,1932,Summer,Los Angeles,Art Competitions,"Art Competitions Mixed Painting, Unknown Event",NaN,USA,NaN


SGP is the new NOC for singapore while SIN is used in the noc_regions table
rot, tuv and unk are na in the noc_regions with rot, tuv having their true names in notes and unk is actually unknown (also in notes column)

In [146]:
missing_region = df[df["region"].isna()]
missing_region["NOC"].value_counts()

NOC
SGP    349
ROT     12
TUV      7
UNK      2
Name: count, dtype: int64

In [148]:
df[df["Age"].isna()]["Year"].describe()

count    9474.000000
mean     1930.829428
std        23.819952
min      1896.000000
25%      1908.000000
50%      1924.000000
75%      1948.000000
max      2008.000000
Name: Year, dtype: float64

years where age had missing values from 1896 to 2008 with 1948 having the highest amount of missing values.

In [149]:
df[df["Age"].isna()]["Year"].value_counts().sort_index()

Year
1896     163
1900     790
1904     274
1906     743
1908     649
1912     156
1920     845
1924    1142
1928     963
1932     330
1936     213
1948    1176
1952     277
1956     638
1960     221
1964      56
1968     118
1972      96
1976      52
1980     187
1984     216
1988     110
1992      44
1994       2
1996       8
1998       2
2000       1
2008       2
Name: count, dtype: int64

In [172]:
df[df["Age"].isna()]["Year"].value_counts().sort_values(ascending=False)

Year
1948    1176
1924    1142
1928     963
1920     845
1900     790
1906     743
1908     649
1956     638
1932     330
1952     277
1904     274
1960     221
1984     216
1936     213
1980     187
1896     163
1912     156
1968     118
1988     110
1972      96
1964      56
1976      52
1992      44
1996       8
2008       2
1994       2
1998       2
2000       1
Name: count, dtype: int64

### Investigating inconsistencies

In [150]:
df.columns

Index(['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games',
       'Year', 'Season', 'City', 'Sport', 'Event', 'Medal', 'region', 'notes'],
      dtype='str')

ID uniquely identifies an athlete across all the dataset. no alternative names/misspelling occured, no misgendering and height and weight were measured once and used for future records as no change in weight even though normally weight changes even if slightly.

In [151]:
id_checks = df.groupby(["ID"])[["Name", "Sex", "Height", "Weight", "NOC"]].nunique(dropna=False)
print((id_checks > 1).sum())

Name         0
Sex          0
Height       0
Weight       0
NOC       1570
dtype: int64


Games column consistent with year and season data

In [152]:
(df["Games"] != df["Year"].astype(str) + " " + df["Season"]).sum()

np.int64(0)

In [153]:
(df.groupby("Event")["Sport"].nunique() > 1).sum()

np.int64(0)

no male athlete participated in a women's event

In [175]:
f"{((df["Event"].str.contains("Women's", case=False, na=False))).sum():,}"

'71,916'

In [176]:
(df["Sex"] == "M").sum().__format__(",")

'196,594'

In [156]:
any((df["Sex"] == "M") & (df["Event"].str.contains("Women's", case=False, na=False)))

False

no female athlete participated in a men's event

In [157]:
(df["Sex"] == "F").sum()

np.int64(74522)

In [158]:
(df["Event"].str.contains(r"\bMen's", case=False, na=False)).sum()

np.int64(182260)

In [159]:
((df["Sex"] == "F") & (df["Event"].str.contains(r"\bMen's", case=False, na=False))).any()

np.False_

SGP instead of SIN

In [160]:
(df["NOC"] == "SGP").sum()

np.int64(349)